# Notebook #2 — osm-train (P1.1 + P1.2 core; continues from the specialist)

This is a **C1 cloud trainer**. It pulls the `typed-decisions` agent from Hugging Face, continues
training it on this device's C2 harness objectives, then hands the new checkpoint back for gating:

1. **Calibration (P1.1-1)** — soft-teacher objective: negative strictly-proper scoring rule
   (`proper_reward`, the same rule family the checkpoints trained on) against the gold teacher
   distributions in `LocalLLaMA/typed-decisions`, plus Banking77 77-way options.
2. **In-loop temperature (P1.1-2)** — every `VAL_STEPS`, fit ECE-minimising temperatures on a
   held-out slice and bake the best fit into the saved `rl_agent_config.json`, so the served model
   uses minimum-ECE bucket temperatures.
3. **Permutation augmentation (P1.2-7)** — every training example re-orders its options (uniform
   shuffles for `choice`, occasional false/true flips for `noul`), supervised against a
   re-aligned teacher. This is the direct counter to the Banking77 `option_order_robustness ~0`.
4. **Context milestone (P1.2-2)** — training at 2k, serving at **4k** (`max_len=4096`,
   `head_max_len=768`): the saved checkpoint evals at 4k context, so the report's latency/recall
   reflect the unlocked window. ModernBERT natively supports 8k; 32k/retrieval-head land in a
   later notebook.

Outputs handed back (`/kaggle/working`):
- `osm-v0/` — self-contained checkpoint (rl_agent_config.json + model.safetensors + encoder/ + tokenizer/)
- `report.jsonl` + `summary.md` — frozen-schema rows for `osm-v0` on the **same**
  `v0-baseline-r2` grid (exact comparison vs the v0-baseline you just ran)
- `telemetry.jsonl` — per-step loss + val ECE/temperature curve
- `probe.jsonl` — repeated SDK check on the freshly trained checkpoint

Requirements: Accelerator **GPU T4 x2**, Internet **On**. Edit the CONFIG cell, Run All (~1.5–2 h).

In [ ]:
# ============ CONFIG ============
REPO_URL = "https://github.com/ahmed-alqasaby/laya.git"
REF = "main"
REPO_DIR = "/kaggle/working/laya"
GRID = "configs/eval_grid.yaml"
OUTDIR = "/kaggle/working"

# base checkpoint (start from the specialist, not the generalist root)
BASE_MODEL = "convaiinnovations/laya"
BASE_SUBFOLDER = "typed-decisions"
TRAIN_OUT_DIR = "/kaggle/working/osm-v0"

# data caps (train rows are heavy; keep the run on T4 pacing)
TD_MAX = 8000        # typed-decisions train rows used for training
TD_VAL = 1000        # held-out type-decisions val rows (excluded from fit)
B77_MAX = 6000       # banking77 train rows (77-way, drives permutation aug)
B77_VAL = 500        # held-out banking77 val rows (excluded from fit)

# training
SEED = 7
EPOCHS = 1
LR = 5e-4
WARMUP_STEPS = 200
W_SHP = 0.5          # proper_reward spherical-weight (soft-teacher smoothing)
W_RPS = 1.0          # proper_reward ranked-probability weight (score only)
MAX_TOKENS_PER_BATCH = 4096   # same token budget the original trainer used
MAX_BATCH_ITEMS = 8
GRAD_CLIP = 1.0
LOG_STEPS = 10
VAL_STEPS = 60                 # in-loop temperature refit + val metrics cadence

# context (P1.2-2): train at shorter context, serve/eval at 4k
TRAIN_MAX_LEN = 2048
TRAIN_HEAD_MAX_LEN = 768
EVAL_MAX_LEN = 4096
EVAL_HEAD_MAX_LEN = 768

# trained-variant name; the eval grid runs it with force_variants so the
# frozen v0-baseline-r2 grid config needs no edits
MODELS = {"osm-v0": (TRAIN_OUT_DIR, None)}
EVAL_VARIANTS = ["osm-v0"]

In [ ]:
import os, subprocess, sys
os.environ["USE_TF"] = "0"  # transformers + TF can deadlock; see laya model card

subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                "laya", "datasets", "scipy", "pyyaml", "psutil", "safetensors"], check=True)

if not os.path.isdir(REPO_DIR):
    subprocess.run(["git", "clone", "--depth", "1", "-b", REF, REPO_URL, REPO_DIR], check=True)
sys.path.insert(0, REPO_DIR)
if not os.path.isdir(os.path.join(REPO_DIR, "layaenv")):
    raise RuntimeError(
        "C2 package layaenv missing in the cloned repo. "
        "Push the C2 work first, then Run All again."
    )

import json, math, copy, random, time, shutil
import numpy as np
import torch
import huggingface_hub

import layaenv
import laya
from laya.common import build_model, build_sequence, collate_items, proper_reward, QTYPES

print("torch", torch.__version__, "| cuda", torch.cuda.is_available(),
      "| device", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "cpu")
if not torch.cuda.is_available():
    raise RuntimeError("this notebook needs a GPU accelerator slot")
torch.manual_seed(SEED)
np.random.seed(SEED)
random.seed(SEED)

In [ ]:
# --- pull the specialist checkpoint and build the identical torch model ---
from huggingface_hub import snapshot_download
from safetensors.torch import load_file
from transformers import AutoTokenizer

BASE_DIR = snapshot_download(BASE_MODEL, allow_patterns=[f"{BASE_SUBFOLDER}/*"])
BASE_DIR = os.path.join(BASE_DIR, BASE_SUBFOLDER)
with open(os.path.join(BASE_DIR, "rl_agent_config.json")) as f:
    BASE_CFG = json.load(f)

print("base:", BASE_CFG.get("model_name"), "| encoder", BASE_CFG["encoder"],
      "| head_layers", BASE_CFG.get("head_layers"), "| act_costs", BASE_CFG.get("act_costs"))

tok = AutoTokenizer.from_pretrained(os.path.join(BASE_DIR, "tokenizer"))
model = build_model(BASE_CFG, encoder_dir=os.path.join(BASE_DIR, "encoder"))
sd = load_file(os.path.join(BASE_DIR, "model.safetensors"))
model.load_state_dict(sd, strict=True)
print("base weights loaded:", len(sd), "tensors")

# freeze the encoder (calibration work is in the decision head) and the act/rejector
# head (that's Notebook 2b; leaving it untouched keeps the checkpoint contract intact)
for p in model.encoder.parameters():
    p.requires_grad = False
for p in model.act_head.parameters():
    p.requires_grad = False
trainable = [n for n, p in model.named_parameters() if p.requires_grad]
print("trainable:", trainable)

# --- sanity forward pass on one question before any expensive setup ---
q = {"t": "choice", "ins": "Which department should handle this request?",
     "crit": {"billing": "invoices", "technical": "bugs", "other": "everything else"}}
seq, markers = build_sequence(tok, {"body": "duplicate charge on invoice #4411"}, q,
                              TRAIN_MAX_LEN, TRAIN_HEAD_MAX_LEN)
b = collate_items([[{"ids": seq, "markers": markers, "qtype": QTYPES[q["t"]]}]], tok.pad_token_id)
with torch.no_grad():
    logits, act = model(b["input_ids"], b["attention_mask"], b["marker_pos"],
                        b["marker_mask"], b["qtype"])
assert logits.shape[-1] == len(markers) == 3, (logits.shape, len(markers))
print("forward pass OK:", tuple(logits.shape))

In [ ]:
# --- training data: typed-decisions (soft teacher) + banking77 (77-way one-hot) ---
import datasets as hf_datasets
from layaenv import datasets as dsx

def _internal(inst):
    q = inst.question
    crit = q.get("criteria", q.get("crit"))
    return {"t": inst.qtype, "ins": q.get("instructions", q.get("ins", "")), "crit": crit}

def _gold_idx(inst):
    if isinstance(inst.answer_idx, int) and inst.answer_idx >= 0:
        return int(inst.answer_idx)
    if inst.ordinal_value is not None and inst.option_levels is not None:
        lv = np.asarray(inst.option_levels, dtype=float)
        return int(np.abs(lv - float(inst.ordinal_value)).argmin())
    return 0

def _target(inst, order):
    k = len(inst.options) if inst.options else len(inst.option_levels if inst.option_levels is not None else [])
    if inst.teacher is not None:
        t = np.asarray(inst.teacher, dtype=np.float32)
    else:
        idx = int(inst.answer_idx) if isinstance(inst.answer_idx, int) and inst.answer_idx >= 0 else 0
        t = np.eye(k, dtype=np.float32)[idx] if k else np.array([1.0], dtype=np.float32)
    if order is not None:
        t = t[order]
    return t

def make_epoch_item(inst, rng):
    q = _internal(inst)
    k = len(inst.options) if inst.options else (len(inst.option_levels) if inst.option_levels is not None else 0)
    order = None
    if inst.qtype == "choice" and k >= 2:
        order = list(range(k))
        rng.shuffle(order)
    elif inst.qtype == "noul":
        order = [1, 0] if rng.random() < 0.5 else [0, 1]
    # score stays ordinal: order is fixed
    return q, order, _target(inst, order)

def prepare_items(insts, shuffle, rng, max_len, head_max_len):
    out = []
    for inst in insts:
        q, order, target = make_epoch_item(inst, rng)
        seq, markers = build_sequence(tok, inst.state, q, max_len, head_max_len, option_order=order)
        if not markers:
            continue
        out.append({"ids": seq, "markers": markers, "qtype": QTYPES[inst.qtype],
                    "target": target.tolist(), "label": _gold_idx(inst), "len": len(seq)})
    if shuffle:
        rng.shuffle(out)
    return out

def pack_batches(items):
    batches, cur, left = [], [], MAX_TOKENS_PER_BATCH
    for it in items:
        if cur and (it["len"] + 16 > left or len(cur) >= MAX_BATCH_ITEMS):
            batches.append(cur)
            cur, left = [], MAX_TOKENS_PER_BATCH
        cur.append(it)
        left -= it["len"] + 16
    if cur:
        batches.append(cur)
    return batches

rng = random.Random(SEED)
td_all = dsx.typed_decisions_rows(list(hf_datasets.load_dataset("LocalLLaMA/typed-decisions", "all", split="train")))
td_all = [i for i in td_all if i.teacher is not None]
print("typed-decisions train rows:", len(td_all))
td_train, td_val = td_all[:TD_MAX], td_all[TD_MAX:TD_MAX + TD_VAL]

b77_all = dsx.banking77_rows(list(hf_datasets.load_dataset("legacy-datasets/banking77", "train")), dsx.BANKING77_LABELS)
b77_train, b77_val = b77_all[:B77_MAX], b77_all[B77_MAX:B77_MAX + B77_VAL]
print("banking77 train rows:", len(b77_all), "| options:", len(dsx.BANKING77_LABELS))

TRAIN = pack_batches(prepare_items(td_train + b77_train, True, rng, TRAIN_MAX_LEN, TRAIN_HEAD_MAX_LEN))
VAL = pack_batches(prepare_items(td_val + b77_val, False, rng, EVAL_MAX_LEN, EVAL_HEAD_MAX_LEN))
print("train batches:", len(TRAIN), "| val batches:", len(VAL), "| val items:", sum(len(x) for x in VAL))
assert sum(len(x) for x in TRAIN) > 0

In [ ]:
# --- training loop: soft-teacher proper scoring rule + permutation aug ---
# (In-loop per-bucket temperature refit on VAL every VAL_STEPS, P1.1-2; baked at save time.)
import torch as T
from transformers import get_linear_schedule_with_warmup
from layaenv.metrics import rows_cal_ece, rows_fit_temperature
from laya.common import temp_bucket

model.train()
opt = T.optim.AdamW([p for n, p in model.named_parameters() if p.requires_grad], lr=LR, weight_decay=0.01)
sched = get_linear_schedule_with_warmup(opt, num_warmup_steps=WARMUP_STEPS,
                                        num_training_steps=max(1, len(TRAIN) * EPOCHS))
device = "cuda:0"
model.to(device)

def softmax_rows(rows):
    probs, gold = [], []
    for lr, mm, gi in rows:
        k = int(mm.sum())
        z = lr.float()[:k]
        p = T.exp((z - z.max()))
        probs.append((p / p.sum()).cpu().numpy())
        gold.append(gi)
    return probs, gold

def run_val():
    model.eval()
    rows_by_bucket = {}
    with T.no_grad():
        for group in VAL:
            b = collate_items([[it] for it in group], tok.pad_token_id)
            ids = b["input_ids"].to(device)
            am = b["attention_mask"].to(device)
            mp = b["marker_pos"].to(device)
            mm = b["marker_mask"].to(device)
            qt = b["qtype"].to(device)
            logits, _ = model(ids, am, mp, mm, qt)
            for r in range(len(group)):
                k = int(mm[r].sum())
                bucket = temp_bucket(int(qt[r]), k)
                rows_by_bucket.setdefault(bucket, []).append(
                    (logits[r], mm[r], int(b["label"][r])))
    probs_all, gold_all, bucket_temps = [], [], {}
    for bucket, rows in rows_by_bucket.items():
        p, g = softmax_rows(rows)
        probs_all += p
        gold_all += g
        if len(p) >= 8:
            t, _ = rows_fit_temperature(p, g)
            if t is not None:
                bucket_temps[bucket] = round(float(t), 5)
    model.train()
    ece = rows_cal_ece(probs_all, gold_all)
    t_mean, post = rows_fit_temperature(probs_all, gold_all)
    return len(probs_all), ece, post, t_mean, bucket_temps, probs_all, gold_all

telemetry = []
best = {"err": float("inf"), "step": -1}
t_start = time.time()
step_done = 0
for epoch in range(EPOCHS):
    for bi, group in enumerate(TRAIN):
        b = collate_items([[it] for it in group], tok.pad_token_id)
        ids = b["input_ids"].to(device)
        am = b["attention_mask"].to(device)
        mp = b["marker_pos"].to(device)
        mm = b["marker_mask"].to(device)
        qt = b["qtype"].to(device)
        target = b["target"].to(device).float()
        msk = mm.float()

        with T.autocast("cuda", dtype=T.float16):
            logits, _ = model(ids, am, mp, mm, qt, detach_encoder=True)
        probs = logits.float().softmax(-1)
        rw = proper_reward(probs, target, qt, msk, w_sph=W_SHP, w_rps=W_RPS, log_floor=-9.21)
        loss = -rw.mean()

        opt.zero_grad(set_to_none=True)
        loss.backward()
        T.nn.utils.clip_grad_norm_([p for p in model.parameters() if p.requires_grad], GRAD_CLIP)
        opt.step()
        sched.step()
        step_done += 1

        if step_done % LOG_STEPS == 0:
            print(f"epoch {epoch} step {step_done}/{len(TRAIN)} loss {loss.item():.4f}", flush=True)

        if step_done % VAL_STEPS == 0:
            nv, ece, post, t_mean, btemps, probs_v, gold_v = run_val()
            slot = {"step": step_done, "epoch": epoch, "loss": round(float(loss.item()), 4),
                    "val_items": nv,
                    "val_raw_ece": None if ece is None else round(ece, 4),
                    "val_postfit_ece": None if post is None else round(post, 4),
                    "val_temp_mean": None if t_mean is None else round(t_mean, 4),
                    "bucket_temps": btemps}
            acc = float(np.mean([int(np.argmax(p) == y) for p, y in zip(probs_v, gold_v)]))
            slot["val_argmax_acc"] = round(acc, 4)
            telemetry.append(slot)
            metric = (1.0 - acc) + (post if post is not None else 1.0)
            if metric < best["err"]:
                best = {"err": metric, "step": step_done}
            print(f"  val n={nv} acc={acc:.3f} rawECE={ece} postECE={post} temp={t_mean} "
                  f"best@step {best['step']} buckets={sorted(btemps)}", flush=True)

hours = (time.time() - t_start) / 3600.0
print("training done:", step_done, "updates in", round(hours, 2), "h; best ckpt at step", best["step"])
with open(os.path.join(OUTDIR, "telemetry.jsonl"), "w") as f:
    for row in telemetry:
        f.write(json.dumps(row) + "\n")


In [ ]:
# --- bake config (4k context + in-loop bucket temps), save, verify, eval ---
FINAL_CFG = json.loads(json.dumps(BASE_CFG))
FINAL_CFG["max_len"] = EVAL_MAX_LEN
FINAL_CFG["head_max_len"] = EVAL_HEAD_MAX_LEN
# in-loop fitted bucket temperatures (P1.1-2): minimum-ECE per bucket, merged
# over the base map. The harness reports its own independent fit too.
bucket_temps = {}
for row in telemetry:
    bucket_temps.update(row.get("bucket_temps") or {})
if bucket_temps:
    merged = dict(FINAL_CFG.get("temperature_by_options") or {})
    merged.update(bucket_temps)
    FINAL_CFG["temperature_by_options"] = merged
tm = [row["val_temp_mean"] for row in telemetry if row.get("val_temp_mean") is not None]
if tm:
    FINAL_CFG["temperature"] = [round(float(np.mean(tm)), 6)] * 3
FINAL_CFG["training"] = {
    "updates": step_done,
    "epochs_completed": epoch + 1,
    "hours": round(hours, 2),
    "world_size": 1,
    "fine_tuned_from_checkpoint": True,
    "base": BASE_SUBFOLDER,
    "objectives": ["soft_teacher_proper_rule", "permutation_aug", "bucket_temp_refit", "ctx_4k"],
    "best_step": best["step"],
}

os.makedirs(TRAIN_OUT_DIR, exist_ok=True)
shutil.copytree(os.path.join(BASE_DIR, "encoder"), os.path.join(TRAIN_OUT_DIR, "encoder"), dirs_exist_ok=True)
shutil.copytree(os.path.join(BASE_DIR, "tokenizer"), os.path.join(TRAIN_OUT_DIR, "tokenizer"), dirs_exist_ok=True)
with open(os.path.join(TRAIN_OUT_DIR, "rl_agent_config.json"), "w") as f:
    json.dump(FINAL_CFG, f, indent=2)
from safetensors.torch import save_file as st_save
st_save({k: v.detach().cpu().contiguous() for k, v in model.state_dict().items()},
        os.path.join(TRAIN_OUT_DIR, "model.safetensors"))
print("checkpoint written ->", TRAIN_OUT_DIR)

# verify the checkpoint reloads through the real laya SDK, then eval on the grid
agent = layaenv.backends.load_backend("osm-v0", TRAIN_OUT_DIR, None)
print("osm-v0 loaded through the real laya SDK on", agent.agent.device)

probe_state = {"body": "Duplicate charge on invoice #4411"}
probe_q = {"department": {"type": "choice", "instructions": "Which department?",
                          "criteria": {"billing": "invoices", "technical": "bugs", "other": "other"}}}
t0 = time.perf_counter()
res = agent.predict(probe_state, probe_q)
dt = (time.perf_counter() - t0) * 1000.0
with open(os.path.join(OUTDIR, "probe.jsonl"), "w") as f:
    f.write(json.dumps({"variant": "osm-v0", "ok": True,
                        "elapsed_ms": round(dt, 1), "result": res}) + "\n")
print("probe:", dt, "ms ->", res["answers"]["department"])

rows, skipped = layaenv.run_grid(
    os.path.join(REPO_DIR, GRID),
    repo_root=REPO_DIR,
    outdir=OUTDIR,
    model_registry=MODELS,
    force_variants=EVAL_VARIANTS,
)
for s in skipped:
    print("SKIP:", s)
print("produced", len(rows), "report rows for osm-v0")

zip_path = layaenv.write_artifacts(OUTDIR, rows, skipped,
                                   extra={"checkpoint": TRAIN_OUT_DIR, "telemetry_rows": len(telemetry)})
print(layaenv.schema.render_metric_table(rows))
import zipfile
with zipfile.ZipFile(zip_path) as z:
    print("artifact contents:", ", ".join(z.namelist()))
print("DONE:", zip_path)